# (5) 인터뷰 피드백 보고서 : 고도화

In [ ]:
from collections import defaultdict

def summarize_interview(state: InterviewState) -> InterviewState:
	# 여기에 코드를 완성합니다.
	print("\n\n===== 인터뷰 피드백 보고서 =====\n")
	
	# 기본 확인
	if not state.get("conversation") and not state.get("evaluation"):
		print("아직 진행된 인터뷰가 없습니다.")
		return state
	
	# evaluation 리스트 확보
	evals = state.get("evaluation", [])
	#print(evals)
	if not isinstance(evals, list) or len(evals) == 0:
		print("evaluation 항목이 비어있거나 형식이 올바르지 않습니다.")
		return state

	# question_strategy별로 그룹화
	groups = defaultdict(list)
	order = []  # 전략 등장 순서 보존용
	for item in evals:
		strat = item.get("question_strategy") or "기타"
		if strat not in groups:
			order.append(strat)
		groups[strat].append(item)
	# display(groups)

	# 모델 선언	
	llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
	prompt = PromptTemplate(
		input_variables=["temp"],
		template=
		"""
		당신은 인사담당 면접관입니다.
		아래의 질문 전략별 문답에 대한 총평을 작성하세요.
		출력 형식은 반드시 아래의 구조를 따르세요.
		불필요한 문장이나 서론 없이 형식 그대로 출력하십시오.

		[피드백 형식]
		[전략명]
		1. 답변 스타일: (지원자의 전반적인 답변 방식 요약)
		2. 강점: (잘한 점을 구체적으로 기술)
		3. 약점: (부족한 점 및 개선 포인트)
		4. 종합 평가: (해당 전략 영역에 대한 총평, 3~5문장 내외)

		예시 출력
		[문제 해결 능력]
		1. 답변 스타일: 논리적으로 원인을 파악하고 해결책을 제시함.
		2. 강점: 문제 상황을 구조적으로 접근함.
		3. 약점: 구체적 수치나 성과 언급이 부족함.
		4. 종합 평가: 전반적으로 논리적이지만 실무적인 구체성이 다소 부족함.

		--- 실제 입력 데이터 ---
		{temp}
    """
	)

	strategy_feedbacks = []
	# 전략별로 출력
	for strat in order:
		temp = ""
		items = groups[strat]
		print(f"--- 전략: {strat} (문항 수: {len(items)}) ---\n")
		for i, qna in enumerate(items, 1):
			question = (qna.get("question") or "").strip()
			answer = (qna.get("answer") or "").strip()
			evaluation = qna.get("evaluation", {})

			print(f"Q{i}. {question}")
			print(f"A. {answer}")

			if isinstance(evaluation, dict) and evaluation:
				print("평가:")
				for k, v in evaluation.items():
					print(f" - {k}: {v}")
			else:
				print("평가: 없음")
			temp += "질문: " + question + '\n답변: ' + answer + '\n총평: ' + evaluation['총평'] + '\n질문 전략: ' + strat
			formatted_prompt = prompt.format(
				temp=temp,
			)
			raw_text = llm.invoke(formatted_prompt).content.strip()
			strategy_feedbacks.append(raw_text)
			print("\n" + raw_text)
			print('\n') 
			

	prompt = PromptTemplate(
    input_variables=["temp"],
    template=
		"""
		당신은 인사담당 면접관입니다.
		아래는 면접에서 평가된 각 질문 전략별 피드백입니다.
		이 내용을 바탕으로 지원자에 대한 **면접 총평**을 작성하세요.
		출력 형식은 반드시 아래의 구조를 따르며, 불필요한 서론 없이 형식 그대로 출력하십시오.

		[출력 형식]
		[면접 총평]
		1. 전반적인 인상: (면접 중 지원자의 태도, 표현력, 커뮤니케이션 등 전반적 인상 요약)
		2. 강점 요약: (각 전략에서 반복적으로 드러난 강점 및 긍정적 특징)
		3. 개선점 요약: (보완이 필요한 부분, 구체적 개선 방향)
		4. 종합 평가: (지원자의 역량 수준 및 조직 적합성에 대한 총괄 의견 — 5문장 이내)

		--- 참고 데이터 (전략별 피드백) ---
		{temp}
		"""
	)

	temp = "\n".join(strategy_feedbacks)

	# 프롬프트 구성
	formatted_prompt = prompt.format(temp=temp)

	# LLM 호출
	overall_feedback = llm.invoke(formatted_prompt).content.strip()

	print("===== 면접 총평 =====\n")
	print(overall_feedback)

	# return 코드는 제공합니다.
	return state

In [109]:
summarize_interview(updated_state)
None



===== 인터뷰 피드백 보고서 =====

--- 전략: 기타 (문항 수: 1) ---

Q1. KT에서의 인턴 경험 중 가장 기억에 남는 프로젝트는 무엇이었고, 그 과정에서 어떤 역할을 했나요?
A. 사진의 품질에 따라서 OCR의 성능이 달라지는 문제점을 만났고 이를 해결하기 위해서 사진이 명확히 비치는지 확인하는 알고리즘을 통해서 사진 가이드라인을 만들어 제공해 OCR의 성능이 일정하게 유지될 수 있도록 개선했습니다.
평가:
 - 질문과의 연관성: 상
 - 답변의 구체성: 상
 - 총평: 지원자는 질문에 명확하게 답변하며, 프로젝트의 핵심 문제와 해결 방안을 구체적으로 설명했습니다. 전반적으로 우수한 답변입니다.

[기타]
1. 답변 스타일: 프로젝트의 문제와 해결 방안을 명확하게 설명함.
2. 강점: 문제 인식 및 해결을 위한 구체적인 접근 방식과 결과를 잘 제시함.
3. 약점: 프로젝트의 결과나 성과에 대한 구체적인 수치나 예시가 부족함.
4. 종합 평가: 전반적으로 명확하고 구체적인 답변이지만, 성과를 수치적으로 나타내는 것이 더 설득력을 높일 수 있을 것으로 보임.


--- 전략: 경력 및 경험 (문항 수: 1) ---

Q1. KT에서의 인턴 경험 중 가장 기억에 남는 프로젝트는 무엇이었고, 그 과정에서 어떤 역할을 했나요?
A. 사진의 품질에 따라서 OCR의 성능이 달라지는 문제점을 만났고 이를 해결하기 위해서 사진이 명확히 비치는지 확인하는 알고리즘을 통해서 사진 가이드라인을 만들어 제공해 OCR의 성능이 일정하게 유지될 수 있도록 개선했습니다.
평가:
 - 질문과의 연관성: 상
 - 답변의 구체성: 상
 - 총평: 지원자는 인턴 경험 중 기억에 남는 프로젝트를 구체적으로 설명하며, 자신의 역할과 기여를 명확히 드러냈습니다. 전반적으로 질문에 잘 부합하는 답변이었습니다.

[경력 및 경험]
1. 답변 스타일: 구체적인 프로젝트 사례를 통해 자신의 기여를 명확히 설명함.
2. 강점: 문제 해결 과정과 결과를 상세히 기술하여 실질적인 기여도

# 보고서 변수 저장 버전

In [ ]:
from collections import defaultdict

def summarize_interview(state: InterviewState) -> InterviewState:
    print("\n\n===== 인터뷰 피드백 보고서 생성 중... =====\n")

    # 기본 확인
    if not state.get("conversation") and not state.get("evaluation"):
        print("아직 진행된 인터뷰가 없습니다.")
        return state

    evals = state.get("evaluation", [])
    if not isinstance(evals, list) or len(evals) == 0:
        print("evaluation 항목이 비어있거나 형식이 올바르지 않습니다.")
        return state

    # question_strategy별로 그룹화
    groups = defaultdict(list)
    order = []  # 전략 등장 순서 보존용
    for item in evals:
        strat = item.get("question_strategy") or "기타"
        if strat not in groups:
            order.append(strat)
        groups[strat].append(item)

    # 모델 선언
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

    # 전략별 피드백용 프롬프트
    strategy_prompt = PromptTemplate(
        input_variables=["temp"],
        template=
        """
        당신은 인사담당 면접관입니다.
        아래의 질문 전략별 문답에 대한 총평을 작성하세요.
        출력 형식은 반드시 아래의 구조를 따르세요.
        불필요한 문장이나 서론 없이 형식 그대로 출력하십시오.

        [피드백 형식]
        [전략명]
        1. 답변 스타일: (지원자의 전반적인 답변 방식 요약)
        2. 강점: (잘한 점을 구체적으로 기술)
        3. 약점: (부족한 점 및 개선 포인트)
        4. 종합 평가: (해당 전략 영역에 대한 총평, 3~5문장 내외)

        --- 실제 입력 데이터 ---
        {temp}
        """
    )

    report_text = "===== 인터뷰 피드백 보고서 =====\n"
    strategy_feedbacks = []

    # 전략별 처리
    for strat in order:
        temp = ""
        items = groups[strat]

        report_text += f"\n--- 전략: {strat} (문항 수: {len(items)}) ---\n\n"

        for i, qna in enumerate(items, 1):
            question = (qna.get("question") or "").strip()
            answer = (qna.get("answer") or "").strip()
            evaluation = qna.get("evaluation", {})

            report_text += f"Q{i}. {question}\n"
            report_text += f"A. {answer}\n"

            if isinstance(evaluation, dict) and evaluation:
                report_text += "평가:\n"
                for k, v in evaluation.items():
                    report_text += f" - {k}: {v}\n"
            else:
                report_text += "평가: 없음\n"

            report_text += "\n"

            temp += f"질문: {question}\n답변: {answer}\n총평: {evaluation.get('총평','없음')}\n질문 전략: {strat}\n\n"

        # LLM을 통한 전략별 총평 생성
        formatted_prompt = strategy_prompt.format(temp=temp)
        raw_text = llm.invoke(formatted_prompt).content.strip()

        strategy_feedbacks.append(raw_text)
        report_text += "" + raw_text + "\n\n"

    # --- 전체 면접 총평 ---
    total_prompt = PromptTemplate(
        input_variables=["temp"],
        template=
        """
        당신은 인사담당 면접관입니다.
        아래는 면접에서 평가된 각 질문 전략별 피드백입니다.
        이 내용을 바탕으로 지원자에 대한 **면접 총평**을 작성하세요.
        출력 형식은 반드시 아래의 구조를 따르며, 불필요한 서론 없이 형식 그대로 출력하십시오.

        [출력 형식]
        [면접 총평]
        1. 전반적인 인상: (면접 중 지원자의 태도, 표현력, 커뮤니케이션 등 전반적 인상 요약)
        2. 강점 요약: (각 전략에서 반복적으로 드러난 강점 및 긍정적 특징)
        3. 개선점 요약: (보완이 필요한 부분, 구체적 개선 방향)
        4. 종합 평가: (지원자의 역량 수준 및 조직 적합성에 대한 총괄 의견 — 5문장 이내)

        --- 참고 데이터 (전략별 피드백) ---
        {temp}
        """
    )

    temp = "\n".join(strategy_feedbacks)
    formatted_prompt = total_prompt.format(temp=temp)
    overall_feedback = llm.invoke(formatted_prompt).content.strip()

    # 누적 출력에 총평 추가
    report_text += "\n" + overall_feedback + "\n"

    print(report_text)

    # state에 결과 저장
    state["strategy_feedbacks"] = strategy_feedbacks
    state["overall_feedback"] = overall_feedback
    state["report_text"] = report_text

    return state


In [123]:
summarize_interview(updated_state)
None



===== 인터뷰 피드백 보고서 생성 중... =====

===== 인터뷰 피드백 보고서 =====

--- 전략: 기타 (문항 수: 1) ---

Q1. KT에서의 인턴 경험 중 가장 기억에 남는 프로젝트는 무엇이었고, 그 과정에서 어떤 역할을 했나요?
A. 사진의 품질에 따라서 OCR의 성능이 달라지는 문제점을 만났고 이를 해결하기 위해서 사진이 명확히 비치는지 확인하는 알고리즘을 통해서 사진 가이드라인을 만들어 제공해 OCR의 성능이 일정하게 유지될 수 있도록 개선했습니다.
평가:
 - 질문과의 연관성: 상
 - 답변의 구체성: 상
 - 총평: 지원자는 질문에 명확하게 답변하며, 프로젝트의 핵심 문제와 해결 방안을 구체적으로 설명했습니다. 전반적으로 우수한 답변입니다.

[기타]
1. 답변 스타일: 지원자는 질문에 대해 명확하고 구체적인 답변을 제공하며, 프로젝트의 핵심 문제와 해결 방안을 잘 설명했습니다.
2. 강점: 문제를 인식하고 이를 해결하기 위한 구체적인 알고리즘을 개발한 점은 매우 긍정적이며, 실질적인 기여를 강조한 부분이 돋보입니다.
3. 약점: 프로젝트의 결과나 성과에 대한 언급이 부족하여, 개선된 성과가 어떤 영향을 미쳤는지 구체화할 필요가 있습니다.
4. 종합 평가: 지원자의 답변은 문제 해결 능력과 실무 경험을 잘 보여주었으나, 프로젝트의 결과와 성과에 대한 명확한 설명이 추가된다면 더욱 강력한 인상을 줄 수 있을 것입니다. 전반적으로 우수한 답변이었습니다.


--- 전략: 경력 및 경험 (문항 수: 1) ---

Q1. KT에서의 인턴 경험 중 가장 기억에 남는 프로젝트는 무엇이었고, 그 과정에서 어떤 역할을 했나요?
A. 사진의 품질에 따라서 OCR의 성능이 달라지는 문제점을 만났고 이를 해결하기 위해서 사진이 명확히 비치는지 확인하는 알고리즘을 통해서 사진 가이드라인을 만들어 제공해 OCR의 성능이 일정하게 유지될 수 있도록 개선했습니다.
평가:
 - 질문과의 연관성: 상
 - 답변의 구체성: 상
 - 총평: 지원자는 인턴